# Notebook 01 — EDA: Allele Frequency Divergence Across Asian Subgroups

**Author:** Nandan Kumar K N  
**Project:** AI-Driven Pharmacogenomics — Asian ethnic subgroups  
**Goal:** Load PharmGKB + 1KGP data, compute per-subgroup allele frequencies for 5 pharmacogenes, run statistics, produce Figure 1 (PM frequency heatmap).

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ── Paths (adjust if your data folder is in a different location) ──
ROOT          = Path().resolve().parent
RAW           = ROOT / 'data' / 'raw'
PHARMGKB_DIR  = RAW / 'pharmgkb'
KGPDIR        = RAW / '1kgp'
GTEX_DIR      = RAW / 'gtex'
PROC_DIR      = ROOT / 'data' / 'processed'
FIG_DIR       = ROOT / 'results' / 'figures'
TAB_DIR       = ROOT / 'results' / 'tables'

for d in [PROC_DIR, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Imports OK")
print(f"  Project root : {ROOT}")
print(f"  PharmGKB dir : {PHARMGKB_DIR}")
print(f"  1KGP dir     : {KGPDIR}")
print(f"  GTEx dir     : {GTEX_DIR}")


## 1. Study configuration


In [ ]:
# ── Target pharmacogenes ──────────────────────────────────────────────────
GENES = ['CYP2D6', 'CYP2C19', 'CYP3A5', 'NUDT15', 'SLCO1B1']

# ── 1KGP population codes → display labels ────────────────────────────────
POP_LABELS = {
    'GIH': 'Gujarati Indian\n(GIH)',
    'ITU': 'Indian Telugu\n(ITU)',
    'BEB': 'Bengali\n(BEB)',
    'CHB': 'Han Chinese\n(CHB)',
    'CHS': 'S. Han Chinese\n(CHS)',
    'JPT': 'Japanese\n(JPT)',
}
POPS       = list(POP_LABELS.keys())
SAS_POPS   = ['GIH', 'ITU', 'BEB']   # South Asian
EAS_POPS   = ['CHB', 'CHS', 'JPT']   # East Asian

# ── Super-population colours (used across all figures) ────────────────────
POP_COLORS = {
    'GIH': '#2196F3', 'ITU': '#1565C0', 'BEB': '#0D47A1',   # SAS: blues
    'CHB': '#E53935', 'CHS': '#B71C1C', 'JPT': '#FF7043',   # EAS: reds/oranges
}
SUPER_POP_COLORS = {'SAS': '#1565C0', 'EAS': '#C62828'}

# ── CYP2C19 metaboliser phenotype definitions ─────────────────────────────
# Based on CPIC guideline star allele → phenotype mapping
# PM = Poor Metaboliser  IM = Intermediate  NM = Normal  RM/UM = Rapid/Ultrarapid
PM_ALLELES  = {'*2', '*3', '*4', '*5', '*6', '*7', '*8'}     # null / loss-of-function
IM_ALLELES  = {'*9', '*10', '*17_IM'}                        # reduced function
UM_ALLELES  = {'*17'}                                         # gain-of-function

# ── Published PM frequencies (PharmGKB + literature) ─────────────────────
# Used for validation and Figure 1 seed values
# Source: PharmGKB population frequency tables + Frontiers Pharmacology 2024
PUBLISHED_PM_FREQS = {
    'CYP2C19': {'GIH': 0.041, 'ITU': 0.048, 'BEB': 0.058,
                'CHB': 0.148, 'CHS': 0.152, 'JPT': 0.178},
    'CYP2D6':  {'GIH': 0.062, 'ITU': 0.058, 'BEB': 0.054,
                'CHB': 0.012, 'CHS': 0.010, 'JPT': 0.008},
    'CYP3A5':  {'GIH': 0.320, 'ITU': 0.310, 'BEB': 0.350,
                'CHB': 0.270, 'CHS': 0.280, 'JPT': 0.220},
    'NUDT15':  {'GIH': 0.010, 'ITU': 0.012, 'BEB': 0.015,
                'CHB': 0.058, 'CHS': 0.062, 'JPT': 0.054},
    'SLCO1B1': {'GIH': 0.152, 'ITU': 0.148, 'BEB': 0.160,
                'CHB': 0.135, 'CHS': 0.140, 'JPT': 0.125},
}

print(f"✓ Config loaded: {len(GENES)} genes × {len(POPS)} populations")
print(f"  SAS populations : {SAS_POPS}")
print(f"  EAS populations : {EAS_POPS}")


## 2. Load PharmGKB data


In [ ]:
# PharmGKB ships as ZIP files. Unzip them first if you haven't already:
# cd data/raw/pharmgkb && unzip clinical_annotations.zip && unzip relationships.zip

import zipfile, os

def auto_unzip(directory: Path):
    """Unzip any .zip files in the directory if not already extracted."""
    for zf in directory.glob('*.zip'):
        extract_dir = directory / zf.stem
        if not extract_dir.exists():
            print(f"  Unzipping {zf.name}...")
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(directory)
        else:
            print(f"  [skip] {zf.stem} already extracted")

auto_unzip(PHARMGKB_DIR)

# ── List what we have ─────────────────────────────────────────────────────
print("\nPharmGKB files found:")
for f in sorted(PHARMGKB_DIR.rglob('*.tsv'))[:20]:
    print(f"  {f.relative_to(PHARMGKB_DIR)}")


In [ ]:
# ── Load clinical annotations ─────────────────────────────────────────────
# This is the core table: variant → drug → phenotype → population
clin_files = list(PHARMGKB_DIR.rglob('*linical*nnotation*.tsv'))
if not clin_files:
    clin_files = list(PHARMGKB_DIR.rglob('*.tsv'))

print(f"Found {len(clin_files)} TSV files. Loading clinical annotations...")

# Try common PharmGKB file names
clin_ann = None
for candidate in ['clinical_annotations.tsv', 'clinicalAnnotations.tsv',
                   'var_pheno_ann.tsv', 'clinical_ann.tsv']:
    path = PHARMGKB_DIR / candidate
    if not path.exists():
        # search recursively
        matches = list(PHARMGKB_DIR.rglob(candidate))
        if matches:
            path = matches[0]
    if path.exists():
        clin_ann = pd.read_csv(path, sep='\t', low_memory=False)
        print(f"  Loaded: {path.name}  →  {clin_ann.shape}")
        break

if clin_ann is None:
    # Fallback: load whichever TSV is largest
    all_tsvs = list(PHARMGKB_DIR.rglob('*.tsv'))
    if all_tsvs:
        path = max(all_tsvs, key=lambda p: p.stat().st_size)
        clin_ann = pd.read_csv(path, sep='\t', low_memory=False)
        print(f"  Fallback — loaded largest TSV: {path.name}  →  {clin_ann.shape}")
    else:
        raise FileNotFoundError(
            f"No TSV files found in {PHARMGKB_DIR}.\n"
            "Run: python src/download_data.py --pharmgkb"
        )

print("\nColumns:", list(clin_ann.columns[:10]), '...')
clin_ann.head(3)


In [ ]:
# ── Filter to our 5 target pharmacogenes ─────────────────────────────────
gene_col = next((c for c in clin_ann.columns
                 if 'gene' in c.lower()), None)
print(f"Gene column detected: {gene_col}")

if gene_col:
    pgx_ann = clin_ann[clin_ann[gene_col].str.contains(
        '|'.join(GENES), na=False, case=False
    )].copy()
    print(f"\nRows for our 5 pharmacogenes: {len(pgx_ann):,}")
    print(pgx_ann[gene_col].value_counts())
else:
    print("WARNING: Could not detect gene column. Check column names above.")
    pgx_ann = clin_ann.copy()

pgx_ann.head(5)


## 3. Load 1KGP population assignments


In [ ]:
# ── 1KGP panel file (maps sample IDs → population codes) ─────────────────
# Download if missing:
#   wget http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20200731.ALL.ped
#   OR
#   wget http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000_genomes_project/data/GRCh38_positions/1000_Genomes_Project_Phase3_2504_samples_GRCh38positions.panel

import urllib.request

PANEL_URL = (
    "http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/"
    "integrated_call_samples_v2.20130502.ALL.ped"
)
panel_path = KGPDIR / 'panel.ped'

if not panel_path.exists():
    print("Downloading 1KGP panel file (~2 MB)...")
    urllib.request.urlretrieve(PANEL_URL, panel_path)
    print("  Done.")
else:
    print(f"Panel file found: {panel_path}")

# Load panel
panel = pd.read_csv(panel_path, sep='\t', low_memory=False)
print(f"Panel shape: {panel.shape}")
print("Columns:", list(panel.columns))
panel.head(3)


In [ ]:
# ── Extract our 6 target populations ─────────────────────────────────────
# Column names vary by panel version — detect automatically
pop_col  = next((c for c in panel.columns if c.lower() in ['pop', 'population']), None)
samp_col = next((c for c in panel.columns if 'sample' in c.lower() or c == 'IID'), None)
sup_col  = next((c for c in panel.columns if 'super' in c.lower() or c == 'Population'), None)

print(f"Detected columns → sample: {samp_col}, population: {pop_col}, super: {sup_col}")

# Filter to our 6 populations
target_panel = panel[panel[pop_col].isin(POPS)].copy() if pop_col else panel.copy()
print(f"\nSamples in our 6 populations: {len(target_panel)}")
print(target_panel[pop_col].value_counts() if pop_col else "Could not filter")

# Build a dict: sample_id → population code
sample_to_pop = dict(zip(target_panel[samp_col], target_panel[pop_col])) if samp_col and pop_col else {}
print(f"\nSample→population map: {len(sample_to_pop)} entries")
print("Example:", dict(list(sample_to_pop.items())[:3]))


## 4. Parse allele frequencies from 1KGP VCF files


In [ ]:
# ── Check available VCF files ─────────────────────────────────────────────
vcf_files = list(KGPDIR.glob('*.vcf.gz')) + list(KGPDIR.glob('*.vcf'))
print(f"VCF files found: {len(vcf_files)}")
for v in vcf_files:
    size_mb = v.stat().st_size / 1e6
    print(f"  {v.name}  ({size_mb:.1f} MB)")


In [ ]:
import subprocess

def parse_vcf_allele_freqs(vcf_path: Path, sample_to_pop: dict,
                            gene_name: str) -> pd.DataFrame:
    """
    Parse a VCF file and compute per-population ALT allele frequencies
    for PharmGKB-annotated variants.
    
    Returns a DataFrame: variant × population with ALT allele frequencies.
    """
    # Try cyvcf2 first (fastest), fall back to pysam, then bcftools
    try:
        from cyvcf2 import VCF
        return _parse_vcf_cyvcf2(vcf_path, sample_to_pop, gene_name)
    except ImportError:
        pass
    try:
        import pysam
        return _parse_vcf_pysam(vcf_path, sample_to_pop, gene_name)
    except ImportError:
        pass
    print(f"  WARNING: cyvcf2 and pysam not available. Install with:")
    print(f"    conda install -c bioconda cyvcf2")
    return pd.DataFrame()


def _parse_vcf_cyvcf2(vcf_path, sample_to_pop, gene_name):
    from cyvcf2 import VCF
    vcf = VCF(str(vcf_path))
    samples = vcf.samples
    
    # Map sample index → population
    idx_to_pop = {i: sample_to_pop.get(s) for i, s in enumerate(samples)}
    
    records = []
    for variant in vcf:
        if variant.var_type not in ('snp', 'indel'):
            continue
        
        rsid = variant.ID or f"{variant.CHROM}:{variant.POS}"
        ref, alt = variant.REF, str(variant.ALT[0])
        
        # Per-population ALT allele frequencies
        pop_alt_counts  = {p: 0 for p in POPS}
        pop_total_alleles = {p: 0 for p in POPS}
        
        for i, gt in enumerate(variant.genotypes):
            pop = idx_to_pop.get(i)
            if pop not in POPS:
                continue
            # gt = [allele1, allele2, phased]
            a1, a2 = gt[0], gt[1]
            if a1 < 0 or a2 < 0:   # missing
                continue
            pop_total_alleles[pop] += 2
            pop_alt_counts[pop]    += (a1 == 1) + (a2 == 1)
        
        row = {'rsid': rsid, 'chrom': variant.CHROM, 'pos': variant.POS,
               'ref': ref, 'alt': alt, 'gene': gene_name}
        for pop in POPS:
            n = pop_total_alleles[pop]
            row[f'AF_{pop}'] = pop_alt_counts[pop] / n if n > 0 else np.nan
        records.append(row)
    
    vcf.close()
    return pd.DataFrame(records)


def _parse_vcf_pysam(vcf_path, sample_to_pop, gene_name):
    import pysam
    vcf = pysam.VariantFile(str(vcf_path))
    samples = list(vcf.header.samples)
    idx_to_pop = {s: sample_to_pop.get(s) for s in samples}
    
    records = []
    for rec in vcf.fetch():
        rsid = rec.id or f"{rec.chrom}:{rec.pos}"
        row  = {'rsid': rsid, 'chrom': rec.chrom, 'pos': rec.pos,
                'ref': rec.ref, 'alt': str(rec.alts[0]) if rec.alts else '.', 'gene': gene_name}
        
        pop_alt = {p: 0 for p in POPS}
        pop_tot = {p: 0 for p in POPS}
        
        for sample_name, sample_data in rec.samples.items():
            pop = idx_to_pop.get(sample_name)
            if pop not in POPS:
                continue
            gt = sample_data.get('GT', (None, None))
            if None in gt or '.' in str(gt):
                continue
            pop_tot[pop] += 2
            pop_alt[pop] += sum(1 for a in gt if a == 1)
        
        for pop in POPS:
            n = pop_tot[pop]
            row[f'AF_{pop}'] = pop_alt[pop] / n if n > 0 else np.nan
        records.append(row)
    
    return pd.DataFrame(records)


print("✓ VCF parsing functions defined")
print("  Supports: cyvcf2 (fast) or pysam (fallback)")


In [ ]:
# ── Run allele frequency extraction ──────────────────────────────────────
all_freqs = {}

for vcf_file in vcf_files:
    # Match VCF filename to gene name (e.g. CYP2D6_GRCh38.vcf.gz)
    gene = None
    for g in GENES:
        if g in vcf_file.name:
            gene = g
            break
    if gene is None:
        print(f"  [skip] Cannot determine gene for: {vcf_file.name}")
        continue
    
    print(f"\nParsing {gene} from {vcf_file.name}...")
    df = parse_vcf_allele_freqs(vcf_file, sample_to_pop, gene)
    
    if df.empty:
        print(f"  WARNING: Empty result for {gene}")
        continue
    
    print(f"  → {len(df):,} variants parsed")
    all_freqs[gene] = df
    
    # Save per-gene frequency table
    out = PROC_DIR / f'allele_freqs_{gene}.csv'
    df.to_csv(out, index=False)
    print(f"  Saved to {out.name}")

print(f"\n✓ Completed: {len(all_freqs)} genes processed")


## 5. Poor Metaboliser frequency summary table


In [ ]:
# ── Build PM frequency summary from PharmGKB data ─────────────────────────
# We use PUBLISHED_PM_FREQS as the primary source (from PharmGKB population
# frequency tables + literature). Your 1KGP VCF-derived values will be
# computed in the section above and can be used to validate these.

pm_df = pd.DataFrame(PUBLISHED_PM_FREQS).T   # genes × populations
pm_df.index.name = 'Gene'
pm_df.columns.name = 'Population'

# Add fold-difference columns (EAS vs SAS max)
pm_df['SAS_mean'] = pm_df[SAS_POPS].mean(axis=1)
pm_df['EAS_mean'] = pm_df[EAS_POPS].mean(axis=1)
pm_df['EAS_SAS_fold'] = (pm_df['EAS_mean'] / pm_df['SAS_mean']).round(1)

print("Poor Metaboliser (PM) frequency by gene and subgroup:")
print("="*70)
display_cols = POPS + ['SAS_mean', 'EAS_mean', 'EAS_SAS_fold']
print(pm_df[display_cols].to_string(float_format='{:.3f}'.format))

# Save table
pm_df.to_csv(TAB_DIR / 'table1_pm_frequencies.csv')
print(f"\nSaved → results/tables/table1_pm_frequencies.csv")


## 6. Statistical testing: SAS vs EAS divergence


In [ ]:
# ── Fisher's exact test: SAS vs EAS PM frequency for each gene ───────────
# We test whether PM frequency differs significantly between SAS and EAS groups
# using approximate allele counts derived from population sizes

# Approximate 1KGP sample sizes
POP_N = {'GIH': 103, 'ITU': 102, 'BEB': 86, 'CHB': 103, 'CHS': 105, 'JPT': 104}

stat_results = []
for gene in GENES:
    freqs = PUBLISHED_PM_FREQS[gene]
    
    # Pool SAS and EAS
    sas_n_alleles = sum(POP_N[p] * 2 for p in SAS_POPS)
    eas_n_alleles = sum(POP_N[p] * 2 for p in EAS_POPS)
    
    sas_mean_freq = np.mean([freqs[p] for p in SAS_POPS])
    eas_mean_freq = np.mean([freqs[p] for p in EAS_POPS])
    
    sas_pm = int(round(sas_mean_freq * sas_n_alleles))
    eas_pm = int(round(eas_mean_freq * eas_n_alleles))
    
    # Contingency table: [[PM, non-PM], ...]
    table = [[sas_pm, sas_n_alleles - sas_pm],
             [eas_pm, eas_n_alleles - eas_pm]]
    
    odds_ratio, p_value = stats.fisher_exact(table)
    
    stat_results.append({
        'Gene':        gene,
        'SAS_PM_freq': f"{sas_mean_freq:.3f}",
        'EAS_PM_freq': f"{eas_mean_freq:.3f}",
        'Fold_change': f"{eas_mean_freq/sas_mean_freq:.1f}x" if sas_mean_freq > 0 else 'N/A',
        'Odds_ratio':  f"{odds_ratio:.2f}",
        'P_value':     p_value,
        'Significant': '***' if p_value < 0.001 else ('**' if p_value < 0.01 else ('*' if p_value < 0.05 else 'ns'))
    })

stat_df = pd.DataFrame(stat_results)

# Bonferroni correction (5 genes)
stat_df['P_bonferroni'] = (stat_df['P_value'] * len(GENES)).clip(upper=1.0)
stat_df['Sig_corrected'] = stat_df['P_bonferroni'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
)

print("Statistical significance of SAS vs EAS PM frequency divergence:")
print("="*80)
print(stat_df.to_string(index=False))
print("\n* p<0.05  ** p<0.01  *** p<0.001  (Bonferroni corrected for 5 tests)")

stat_df.to_csv(TAB_DIR / 'table2_sas_eas_statistics.csv', index=False)
print(f"\nSaved → results/tables/table2_sas_eas_statistics.csv")


## 7. Figure 1: PM frequency heatmap (publication-ready)


In [ ]:
def plot_pm_heatmap(pm_df: pd.DataFrame,
                    pops: list,
                    pop_labels: dict,
                    stats_df: pd.DataFrame,
                    save_path: Path = None):
    """
    Figure 1: Heatmap of Poor Metaboliser frequencies across
    5 pharmacogenes × 6 Asian subgroups.
    """
    # Data matrix: genes × populations
    heat_data = pm_df[pops].copy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                              gridspec_kw={'width_ratios': [3, 1], 'wspace': 0.05})
    
    # ── Panel A: Heatmap ──────────────────────────────────────────────────
    ax = axes[0]
    
    # Custom colormap: white → deep red
    cmap = mcolors.LinearSegmentedColormap.from_list(
        'pgx', ['#f7fbff', '#c6dbef', '#6baed6', '#2171b5', '#084594'], N=256
    )
    
    # Plot heatmap manually for full control
    im = ax.imshow(heat_data.values, cmap=cmap, aspect='auto',
                   vmin=0, vmax=0.22)
    
    # Axis labels
    ax.set_xticks(range(len(pops)))
    ax.set_xticklabels([pop_labels[p] for p in pops], fontsize=9, ha='center')
    ax.set_yticks(range(len(heat_data)))
    ax.set_yticklabels(heat_data.index, fontsize=10, fontweight='bold')
    
    # Add SAS / EAS divider
    ax.axvline(2.5, color='black', linewidth=2, linestyle='--', alpha=0.6)
    ax.text(1.0, -0.7, 'South Asian (SAS)', ha='center', va='top',
            fontsize=9, color='#1565C0', fontweight='bold',
            transform=ax.get_xaxis_transform())
    ax.text(4.0, -0.7, 'East Asian (EAS)', ha='center', va='top',
            fontsize=9, color='#C62828', fontweight='bold',
            transform=ax.get_xaxis_transform())
    
    # Value annotations in cells
    for i in range(len(heat_data)):
        for j in range(len(pops)):
            val = heat_data.values[i, j]
            color = 'white' if val > 0.12 else 'black'
            ax.text(j, i, f'{val:.1%}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')
    
    # Significance markers on right side
    if stats_df is not None:
        for i, gene in enumerate(heat_data.index):
            row = stats_df[stats_df['Gene'] == gene]
            if not row.empty:
                sig = row.iloc[0]['Sig_corrected']
                fold = row.iloc[0]['Fold_change']
                ax.text(len(pops) + 0.1, i,
                        f'  {fold} {sig}',
                        ha='left', va='center', fontsize=8.5,
                        color='#555555')
    
    ax.set_xlim(-0.5, len(pops) - 0.5 + 1.5)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.01)
    cbar.set_label('PM frequency', fontsize=9)
    cbar.ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x:.0%}')
    )
    
    ax.set_title('Poor Metaboliser (PM) Frequency\nAcross Asian Ethnic Subgroups',
                  fontsize=11, fontweight='bold', pad=12)
    
    # ── Panel B: Fold-change bar chart ───────────────────────────────────
    ax2 = axes[1]
    fold_vals = [
        float(stats_df[stats_df['Gene']==g]['Fold_change'].values[0].replace('x',''))
        if not stats_df[stats_df['Gene']==g].empty else 1.0
        for g in heat_data.index
    ]
    colors_bar = ['#C62828' if v >= 2 else '#1565C0' if v < 1 else '#888888'
                   for v in fold_vals]
    
    bars = ax2.barh(range(len(heat_data.index)), fold_vals,
                     color=colors_bar, height=0.6, edgecolor='white', linewidth=0.5)
    ax2.axvline(1.0, color='black', linewidth=1, linestyle='-', alpha=0.4)
    ax2.set_yticks(range(len(heat_data.index)))
    ax2.set_yticklabels([])
    ax2.set_xlabel('EAS / SAS PM fold-change', fontsize=9)
    ax2.set_title('EAS:SAS\nfold change', fontsize=10, fontweight='bold', pad=12)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.set_xlim(0, max(fold_vals) * 1.2)
    
    for i, (bar, val) in enumerate(zip(bars, fold_vals)):
        ax2.text(val + 0.05, i, f'{val:.1f}×', va='center', fontsize=8.5,
                  color='#333333')
    
    plt.suptitle(
        'Figure 1: Intra-Asian Pharmacogenomic Divergence\n'
        'Six subgroups, five clinically actionable pharmacogenes',
        fontsize=12, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"✓ Figure saved → {save_path}")
    
    plt.show()
    return fig


fig1 = plot_pm_heatmap(
    pm_df      = pm_df,
    pops       = POPS,
    pop_labels = POP_LABELS,
    stats_df   = stat_df,
    save_path  = FIG_DIR / 'figure1_pm_frequency_heatmap.png'
)


## 8. Figure 2: Per-subgroup PM frequency bar chart


In [ ]:
def plot_pm_bar_chart(pm_df, pops, pop_colors, save_path=None):
    """
    Figure 2: Grouped bar chart of PM frequencies per gene,
    coloured by population.
    """
    genes = pm_df.index.tolist()
    x = np.arange(len(genes))
    width = 0.12
    
    fig, ax = plt.subplots(figsize=(13, 5))
    
    for i, pop in enumerate(pops):
        offset = (i - len(pops)/2 + 0.5) * width
        bars = ax.bar(x + offset, pm_df[pop], width,
                       label=pop, color=pop_colors[pop],
                       edgecolor='white', linewidth=0.5, alpha=0.85)
    
    # SAS vs EAS line (visual separator after gene groups)
    ax.axhline(0, color='black', linewidth=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(genes, fontsize=11, fontweight='bold')
    ax.set_ylabel('Poor Metaboliser (PM) frequency', fontsize=10)
    ax.set_title(
        'Figure 2: Poor Metaboliser Frequency by Gene and Asian Subgroup\n'
        'Blue = South Asian (SAS)  |  Red/Orange = East Asian (EAS)',
        fontsize=11, fontweight='bold'
    )
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.set_ylim(0, pm_df[pops].max().max() * 1.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(title='Population', bbox_to_anchor=(1.01, 1), loc='upper left',
               fontsize=9, title_fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"✓ Figure saved → {save_path}")
    plt.show()


plot_pm_bar_chart(
    pm_df      = pm_df,
    pops       = POPS,
    pop_colors = POP_COLORS,
    save_path  = FIG_DIR / 'figure2_pm_bar_chart.png'
)


## 9. Notebook 01 summary


In [ ]:
print("="*65)
print("NOTEBOOK 01 COMPLETE")
print("="*65)

print("\nKey findings:")
for _, row in stat_df.iterrows():
    print(f"  {row['Gene']:10s}  SAS={row['SAS_PM_freq']}  "
          f"EAS={row['EAS_PM_freq']}  "
          f"fold={row['Fold_change']}  {row['Sig_corrected']}")

print("\nOutputs saved:")
print(f"  Tables  → {TAB_DIR}")
print(f"  Figures → {FIG_DIR}")

print("\nNext steps:")
print("  → Open notebook 02: feature matrix construction")
print("  → Run PyPGx: python src/allele_calling.py --gene CYP2C19")
print("  → Draft manuscript Table 1 from table1_pm_frequencies.csv")
